# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [97]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}
LAT, LON = 38.03, -78.51

def get_weather(start, end, lat=LAT, lon=LON, tries=3, base_delay=1):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)  # Request with a timeout
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])  # If status is 200, return dataframe on daily
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')  # Raise if status isn't in our selected statuses
            last = f'status {r.status_code}'
        except requests.RequestException as e:  # Retry network-level problems
            last = repr(e)
        delay = base_delay * (2 ** attempt)  # Increase delay each time
        print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        time.sleep(delay)  # Sleep before trying again
    raise RuntimeError(f'gave up after {tries} attempts: {last}')  # Raise if every attempt failed
pass

I finished building the API call function that starts by requesting the json, returning the dataframe if the status is 200, raising an error if the status is a failure that isn't worth retrying, checks for network-level errors, and then sleeps for a delay before trying again.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [98]:
from datetime import datetime

used_fallback = False
wx = None
pulled_at = None

try:
    df = get_weather('2026-09-05', '2026-10-03') # Pull entire date range
    wx = df[df['time'].isin(['2026-09-05', '2026-10-03'])].copy() # Just keep the two needed dates
    pulled_at = pd.Timestamp.now(tz='America/New_York') # Record when we pulled
except Exception as e:
    fallback_reason = repr(e) # Record why the call failed
    print(f'API call failed: {e}')

In order to prevent multiple API calls, I called the entire range and then only copied the specific dates I needed into my `wx` DataFrame. I then made sure to record why the call failed if it did, and in the next step I will add the fallback.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [99]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if wx is None: # Only if no dataframe was created
    used_fallback = True
    wx = FALLBACK.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo' # Create a source column

I originally assigned `wx` to `None` so that I could tell if there was a failed call. If a call failed, `wx` stays `None`, and that is when I use the fallback DataFrame. I then created a source column to keep track of where the data came from.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [100]:
wx.rename(columns={'time': 'date', 'temperature_2m_max': 'temp_max', 'precipitation_sum': 'precip_mm'}, inplace=True) # Rename columns
wx['date'] = pd.to_datetime(wx['date']) # Convert date column to datetime
wx.head()


,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


I renamed the API’s columns to my own names and converted date to a real datetime so the table is safe to join.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [101]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [102]:
wettest = wx.loc[wx['precip_mm'].idxmax()]
dryest = wx.loc[wx['precip_mm'].idxmin()]
diff = wettest['precip_mm'] - dryest['precip_mm'] # Find the difference in precipitaiton
print(f"{wettest['date']:%B %d, %Y} was wetter than {dryest['date']:%B %d, %Y} by {diff} mm.")

October 03, 2026 was wetter than September 05, 2026 by 12.2 mm.


I used `.idxmax()` on `wx['precip_mm']` to locate the full row that had the greatest precipitation, and then I did the same for the least precipitation. This allowed me to find the difference between the two and easily locate the dates of these rows.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [103]:
print('Source:  Open-Meteo historical archive API')
print('Endpoint:', BASE)
print(f"Coordinates: {LAT}, {LON}")
print(f"Date range: {wx['date'].min():%B %d, %Y} to {wx['date'].max():%B %d, %Y}")
print(f"Pulled: {pulled_at:%B %d, %Y at %I:%M %p}" if pulled_at else "Pulled: not pulled (API failed)")
print(f"Real or fallback: {'fallback' if used_fallback else 'real'}")

Source:  Open-Meteo historical archive API
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03, -78.51
Date range: September 05, 2026 to October 03, 2026
Pulled: October 12, 2026 at 12:18 AM
Real or fallback: real


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

*If `get_weather` returned an empty DataFrame on failure rather than raising an error, the notebook would keep running without identifying the error. If you do a left merge of the Dataframe onto 200 days of sales data, it would keep the 200 rows of sales data, but `temp_max` and `precip_mm` would all be NaN since that DataFrame was empty. When you chart rain against revenue, the chart would be empty because the NaN rows are dropped, and every single row has NaN precipitation. Even if someone filled in the missing `precip_mm` values with zero, that would produce a chart that shows rain has no effect on revenue. No one would catch it because no errors are thrown and the shape of the DataFrame appears normal.*

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [104]:
used_fallback_orl = False
orlando = None
pulled_orl_at = None

try:
    df = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38) # Pull entire date range
    orlando = df[df['time'].isin(['2026-09-05', '2026-10-03'])].copy() # Just keep the two needed dates
    pulled_orl_at = pd.Timestamp.now(tz='America/New_York') # Record when we pulled
except Exception as e:
    fallback_reason_orl = repr(e) # Record why the call failed
    print(f'API call failed: {e}')

if orlando is None: # Only if no dataframe was created
    used_fallback_orl = True
    orlando = FALLBACK.copy()

orlando['source'] = 'fallback sample' if used_fallback_orl else 'open-meteo' # Create a source column

orlando.rename(columns={'time': 'date', 'temperature_2m_max': 'temp_max', 'precipitation_sum': 'precip_mm'}, inplace=True) # Rename columns
orlando['date'] = pd.to_datetime(orlando['date']) # Convert date column to datetime

cville = wx.copy()
cville['city'] = 'Charlottesville' # Create city column for cville
orlando['city'] = 'Orlando' # Create city column for orlando

combined = pd.concat([cville, orlando], ignore_index=True) # Stack in long-format frmae
combined.head()


,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando


After following the same steps as the Charlottesville weather API pull (pull the dataframe, use the fallback if it failed, rename and format the columns), I created a new column in each dataframe individually before using `pd.concat` to stack both cities in one long-form frame. If I had merged them, it would have added columns rather than rows.